# Garbage Classification (Multimodal)
## ENSF 617 - A2

Fusion learning with ResNet (for image) and DistilBERT (for text).

### Pipeline Overview

![alt text](<Block Diagram.png>)

Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import numpy as np
import os
import re

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Paths/Parameters/Settings Definitions

In [ ]:
BATCH_SIZE = 32
FINE_TUNE_BATCH_SIZE = 16
NUM_WORKERS = 2
NUM_CLASSES = 4
EPOCHS = 10
UNFREEZE_EPOCH = 5
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
SAVE_MODEL_PATH = "/home/le.song/Assignment2/best_image_model.pth"
MISCLASS_FILE = "/home/le.song/Assignment2/misclassified.txt"
LOSS_PLOT_PATH = "/home/le.song/Assignment2/loss_curve.png"
TRAIN_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Train"
VAL_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Val"
TEST_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Test"

Data Transforms

In [ ]:
train_transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.RandomHorizontalFlip(),
transforms.RandomRotation(15),
transforms.ColorJitter(brightness=0.2, contrast=0.2), #Optional, will monitor effects.
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])  #Stats need to align with that of resnet, no stat calculations needed on input data.

test_transform = transforms.Compose([
  transforms.Resize((224, 224)),  #Testing data MUST NOT be pre-processed. No rotations/flips/jitter applied.
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

Datasets and Data Loaders

In [ ]:
import shutil # Added for removing directories

def remove_ipynb_checkpoints(root_dir):
    if not os.path.exists(root_dir):
        print(f"Warning: Root directory {root_dir} does not exist. Skipping cleanup.")
        return
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if '.ipynb_checkpoints' in dirnames:
            checkpoints_path = os.path.join(dirpath, '.ipynb_checkpoints')
            print(f"Removing .ipynb_checkpoints directory: {checkpoints_path}")
            shutil.rmtree(checkpoints_path)
            # Remove from dirnames so os.walk doesn't try to enter it after deletion
            dirnames.remove('.ipynb_checkpoints')

    # Applied as cleanup before creating ImageFolder datasets
remove_ipynb_checkpoints(TRAIN_PATH)
remove_ipynb_checkpoints(VAL_PATH)
remove_ipynb_checkpoints(TEST_PATH)


image_train_ds = ImageFolder(root=TRAIN_PATH, transform=train_transform)
image_val_ds   = ImageFolder(root=VAL_PATH, transform=test_transform)
image_test_ds  = ImageFolder(root=TEST_PATH, transform=test_transform)

train_loader = DataLoader(image_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(image_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(image_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#Verifying sizes/dimension
print("Train samples:", len(image_train_ds))
print("Val samples:", len(image_val_ds))
print("Test samples:", len(image_test_ds))

## Image Classification Model

ResNET Image Encoder Definition

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ImageEncoder(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, proj_dim=256):
        super().__init__()

        # Load pretrained ResNet50
        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        # Remove final classification layer
        self.feature_dim = self.backbone.fc.in_features  # 2048 for ResNet50
        self.backbone.fc = nn.Identity()

        # Optional freezing
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Projection layer (aligns with text embedding size)
        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU()
        )

    def forward(self, x):
        features = self.backbone(x)      # [B, 2048]
        projected = self.projection(features)  # [B, proj_dim]
        return projected


Image Classifier Definition

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, encoder, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, num_classes)
        )


    def forward(self, x):
        embeddings = self.encoder(x)
        logits = self.classifier(embeddings)
        return logits

Initialize Image Model

In [ ]:
encoder = ImageEncoder(pretrained=True, freeze_backbone=True, proj_dim=256)
model = ImageClassifier(encoder).to(device)

Training Loop Definition

In [ ]:
def train_model(model, train_loader, val_loader, device,
                epochs=EPOCHS,               # initial # of epochs used for model initialization
                unfreeze_epoch=UNFREEZE_EPOCH,  # epoch after which the backbone is unfrozen
                initial_lr=INITIAL_LR,       # learning rate before unfreeze
                fine_tune_lr=FINE_TUNE_LR,   # learning rate after unfreeze
                save_path=SAVE_MODEL_PATH):  # file path to save best val-loss model, Saving to FILE may be required, the model will be used later in multi-modal training once combined with text portion.

    # ---- Loss ----
    criterion = nn.CrossEntropyLoss()

    # ---- Initial optimizer: AdamW ----
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=initial_lr,
        weight_decay=1e-4
    )

    train_losses, val_losses = [], []
    best_val_loss = float("inf")

    for epoch in range(epochs):

        # This is how the automatic unfreeze works, once it reaches the defined epoch, backbone unfreezes and AdamW's lr is changed to fine_tune_lr.
        if epoch == unfreeze_epoch:
            print("Unfreezing backbone...")
            for param in model.encoder.backbone.parameters():
                param.requires_grad = True # Unfreeze happens

            # Updating lr in AdamW
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=fine_tune_lr,
                weight_decay=1e-4
            )

            # Reinitialize loader with smaller batch size
            train_loader = DataLoader(
                image_train_ds,
                batch_size=FINE_TUNE_BATCH_SIZE,
                shuffle=True,
                num_workers=4,
                pin_memory=True
            )

        #Training
        model.train()
        running_train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad() #resets gradient between batches
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward() #Back-propagation
            optimizer.step() #Update Weights
            running_train_loss += loss.item()
        
        epoch_train_loss = running_train_loss / len(train_loader) #training loss
        train_losses.append(epoch_train_loss)

        #Validation
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
        epoch_val_loss = running_val_loss / len(val_loader) #validation loss
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{epochs} " #Prints losses
            f"Train Loss: {epoch_train_loss:.4f} "
            f"Val Loss: {epoch_val_loss:.4f}")

        # Saving model with smallest val_loss
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), save_path)
            print("Saved best model.")

    return train_losses, val_losses


Evaluation and Error Logging

In [ ]:
def evaluate_and_log_errors(model, test_loader, device,
                            save_model_path=SAVE_MODEL_PATH,
                            error_log_file=MISCLASS_FILE):

    model.load_state_dict(torch.load(save_model_path))
    model.eval()

    incorrect_samples = []
    correct, total = 0, 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0) #Automatic resizing
            correct += (preds == labels).sum().item()

            # Log misclassified images
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    dataset_index = batch_idx * test_loader.batch_size + i
                    path, _ = test_loader.dataset.samples[dataset_index]
                    incorrect_samples.append(
                        f"{path} | Pred: {preds[i].item()} | True: {labels[i].item()}"
                    )

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")

    with open(error_log_file, "w") as f:
        for line in incorrect_samples:
            f.write(line + "\n")
    print(f"Misclassified samples written to {error_log_file}")
    return accuracy


Loss Curve

In [ ]:
def plot_losses(train_losses, val_losses, save_path=LOSS_PLOT_PATH):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training and Validation Loss")
    plt.savefig(save_path)
    plt.close()
    print(f"Loss curve saved to {save_path}")


Run Image Model

In [ ]:

train_losses, val_losses = train_model(model, train_loader, val_loader, device)
plot_losses(train_losses, val_losses)
accuracy = evaluate_and_log_errors(model, test_loader, device)
imagemodel = model


## Text Classification Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DistilBertForSequenceClassification
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Tokenizer and Model

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)

base_model = base_model.to(device)

Data Extraction from File Names

In [ ]:
import os
import re
import numpy as np

def read_text_files_with_labels(path):
    texts = []
    labels = []

    class_folders = sorted([
    f for f in os.listdir(path)
    if os.path.isdir(os.path.join(path, f)) and not f.startswith(".")
]) #Avoids hidden files being picked up

    label_map = {class_name: idx for idx, class_name in enumerate(class_folders)}

    for class_name in class_folders:
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            file_names = sorted(os.listdir(class_path))

            for file_name in file_names:
                file_path = os.path.join(class_path, file_name)

                if os.path.isfile(file_path):
                    # Remove extension
                    file_name_no_ext, _ = os.path.splitext(file_name)

                    # Replace "_" with space
                    text = file_name_no_ext.replace('_', ' ')

                    # Remove digits
                    text_without_digits = re.sub(r'\d+', '', text)

                    texts.append(text_without_digits)
                    labels.append(label_map[class_name])

    return np.array(texts), np.array(labels)


DATA_ROOT = "/work/TALC/ensf617_2026w/garbage_data"

train_texts, train_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "CVPR_2024_dataset_Train"))
val_texts, val_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "CVPR_2024_dataset_Val"))
test_texts, test_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "CVPR_2024_dataset_Test"))

print("Train samples:", len(train_texts))
print("Val samples:", len(val_texts))
print("Test samples:", len(test_texts))

Text Dataset

In [ ]:
MAX_LEN = 128
UNFREEZE_EPOCH = 5
fine_tune_lr = 2e-5

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
            "text": text
        }


Data Loaders

In [ ]:
BATCH_SIZE = 32

text_train_ds = TextDataset(train_texts, train_labels, tokenizer)
text_val_ds   = TextDataset(val_texts, val_labels, tokenizer)
text_test_ds  = TextDataset(test_texts, test_labels, tokenizer)


train_loader = DataLoader(text_train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(text_val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(text_test_ds, batch_size=BATCH_SIZE)


Text Classifier Definition

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, num_classes, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size
        self.fc = nn.Linear(hidden_size, num_classes)


    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling
        last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask  # (batch, hidden_size)
        return self.fc(pooled)


Text Model Instantiation

In [ ]:
model = TextClassifier(num_classes=4, base_model=base_model).to(device)
#Freeze backbone initially
for param in model.base_model.parameters():
    param.requires_grad = False


Optimizer and Loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)


Text Model Training Loop

In [ ]:
EPOCHS = 10

train_losses = []
val_losses = []
misclassified_files = []

for epoch in range(EPOCHS):

    if epoch == UNFREEZE_EPOCH:
        print("Unfreezing DistilBERT backbone...")

        for param in model.base_model.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=fine_tune_lr
        )

        train_loader = DataLoader(text_train_ds, batch_size=8, shuffle=True)

    # Training

    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)

    # Validation

    model.eval()
    val_loss_total = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            file_names = batch.get("file_name", [None]*labels.size(0))  # fallback if not present

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss_total += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss_total / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)

    # Logging

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print("-" * 40)


Text Model Testing Block

In [ ]:
model.eval()
all_preds = []
misclassified_texts = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())

        # Capture misclassified samples
        mis_idx = (preds != labels).nonzero(as_tuple=True)[0]
        for idx in mis_idx:
            misclassified_texts.append(batch["text"][idx])  # assuming your dataset returns 'text'

# Compute accuracy
accuracy = (torch.tensor(all_preds) == torch.tensor(test_labels)).float().mean()
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Number of misclassified samples: {len(misclassified_texts)}")


Loss Chart

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(range(1, EPOCHS+1), train_losses, label="Train Loss")
plt.plot(range(1, EPOCHS+1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig("/home/le.song/Assignment2/ls_chart_part2")


Save Model

In [ ]:
torch.save(model.state_dict(), "/home/le.song/Assignment2/clean_text_classifier.pth")


## Multi-Modal Model

Dataset

In [ ]:
class MultiModalDataset(Dataset):
    def __init__(self, image_dataset, texts, labels, tokenizer, max_len=128):
        self.image_dataset = image_dataset
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image, _ = self.image_dataset[idx]  # image is already a tensor
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "image": image,  # <-- must be a torch.Tensor
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


MultiModal Model

In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, image_encoder, text_base_model, num_classes=4):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_base_model

        fusion_input_dim = 256 + text_base_model.config.hidden_size

        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, images, input_ids, attention_mask):
        # Image embedding
        img_emb = self.image_encoder(images)

        # Text embedding
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        last_hidden = text_outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        text_emb = (last_hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)

        # Concatenate and pass through fusion head
        x = torch.cat([img_emb, text_emb], dim=1)
        return self.fusion_head(x)



DataLoaders

In [ ]:
train_mm = MultiModalDataset(image_train_ds, train_texts, train_labels, tokenizer)
val_mm   = MultiModalDataset(image_val_ds, val_texts, val_labels, tokenizer)
test_mm  = MultiModalDataset(image_test_ds, test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_mm, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_mm, batch_size=16)
test_loader  = DataLoader(test_mm, batch_size=16)

Initializer MultiModal Model

In [ ]:
multimodal_model = MultiModalModel(encoder, base_model, num_classes=4).to(device)

# Freeze backbones initially
for param in multimodal_model.image_encoder.backbone.parameters():
    param.requires_grad = False
for param in multimodal_model.text_encoder.parameters():
    param.requires_grad = False


Train and Evaluate Skeleton

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, multimodal_model.parameters()),
    lr=1e-3
)

EPOCHS = 12
UNFREEZE_IMAGE_EPOCH = 5
UNFREEZE_TEXT_EPOCH = 8

train_losses, val_losses = [], []

for epoch in range(EPOCHS):

    # Stage 2: Unfreeze image
    if epoch == UNFREEZE_IMAGE_EPOCH:
        for param in multimodal_model.image_encoder.backbone.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, multimodal_model.parameters()),
            lr=1e-4
        )

    # Stage 3: Unfreeze text
    if epoch == UNFREEZE_TEXT_EPOCH:
        for param in multimodal_model.text_encoder.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, multimodal_model.parameters()),
            lr=2e-5
        )

    # ---- Training ----
    multimodal_model.train()
    running_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)              # torch.Tensor
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = multimodal_model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(multimodal_model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ---- Validation ----
    multimodal_model.eval()
    val_loss_total = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = multimodal_model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss_total += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss_total / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}")
    

Plot Loss

In [ ]:
# Plotting Multimodal Loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.axvline(x=UNFREEZE_IMAGE_EPOCH, color='r', linestyle='--', label='Unfreeze Image')
plt.axvline(x=UNFREEZE_TEXT_EPOCH, color='g', linestyle='--', label='Unfreeze Text')
plt.title('Multimodal Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig('/home/le.song/Assignment2/multimodal_loss_curve.png')
plt.show()

Error Evalution

In [ ]:
def evaluate_multimodal_errors(model, test_loader, device, log_file="/home/le.song/Assignment2/mm_misclassified.txt"):
    model.eval()
    incorrect_samples = []
    correct, total = 0, 0

    # We use the original image dataset samples to get the file paths
    # The order is preserved in our MultiModalDataset
    image_paths = test_loader.dataset.image_dataset.samples

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images, input_ids, attention_mask)
            preds = outputs.argmax(dim=1)
            
            total += labels.size(0)
            correct += (preds == labels).sum().item()

            # Identify misclassifications in this batch
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    # Calculate global index to find the path
                    global_idx = batch_idx * test_loader.batch_size + i
                    path, _ = image_paths[global_idx]
                    
                    error_entry = (
                        f"Path: {path} | "
                        f"Pred: {preds[i].item()} | "
                        f"True: {labels[i].item()}"
                    )
                    incorrect_samples.append(error_entry)

    accuracy = correct / total
    print(f"\nFinal Multimodal Test Accuracy: {accuracy:.4f}")

    with open(log_file, "w") as f:
        for line in incorrect_samples:
            f.write(line + "\n")
    print(f"Misclassified multimodal samples logged to {log_file}")
    
    return accuracy


In [ ]:
# Execute evaluation
mm_accuracy = evaluate_multimodal_errors(multimodal_model, test_loader, device)